In [ ]:
import subprocess
import sys
from pathlib import Path

CODE_REVISION = 'c8ba8eea829be0339e8d7757f8ca52dac06e1e32'
MODEL_REVISION = '7476824bc222e4ad509d286e8cae8b8d3f371129'
REPOSITORY = Path('/content/Audio8-ASR-Infinite')
ENVIRONMENT = Path('/content/audio8-venv')
PYTHON = ENVIRONMENT / 'bin/python'
MODEL_FOLDER = Path('/content/audio8-model')
INDEXED_FOLDER = Path('/content/audio8-model-indexed')

def run_command(arguments):
    result = subprocess.run(
        [str(value) for value in arguments], capture_output=True, text=True
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()
    return result.stdout.strip()

def run_audio8(code, *arguments):
    return run_command([PYTHON, '-u', '-c', code, *arguments])

run_command([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])

if PYTHON.exists():
    print('using the existing audio8 environment')
elif ENVIRONMENT.exists():
    raise RuntimeError('environment folder exists but python is missing; inspect before replacing')
else:
    run_command([sys.executable, '-m', 'uv', 'python', 'install', '3.12'])
    run_command([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', ENVIRONMENT])

run_audio8('import sys; print(sys.version); assert sys.version_info[:2] == (3, 12)')
print('environment ready')
model = globals().get('model')


In [ ]:
if not REPOSITORY.exists():
    run_command([
        'git', 'clone', 'https://github.com/Edge0-AI/Audio8-ASR-Infinite.git',
        REPOSITORY,
    ])

changes = run_command(['git', '-C', REPOSITORY, 'status', '--porcelain'])
if changes:
    raise RuntimeError('source checkout has local changes; keep them safe before continuing')

run_command(['git', '-C', REPOSITORY, 'fetch', 'origin', CODE_REVISION])
run_command(['git', '-C', REPOSITORY, 'checkout', '--detach', CODE_REVISION])
run_command([
    sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON,
    '-e', f'{REPOSITORY}[client]',
])
run_command([sys.executable, '-m', 'uv', 'pip', 'check', '--python', PYTHON])

run_audio8('''
import torch
import transformers
import audio8_asr_infinite
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('cuda available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('select a GPU runtime before continuing')
print('gpu:', torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info()
print('free gpu memory:', round(free / 1024**3, 2), 'GiB')
print('total gpu memory:', round(total / 1024**3, 2), 'GiB')
''')
print('installed packages for experiment notes:')
run_command([sys.executable, '-m', 'uv', 'pip', 'freeze', '--python', PYTHON])


In [ ]:
run_audio8('''
import sys
from huggingface_hub import snapshot_download

revision, destination = sys.argv[1:3]
print('model revision:', revision, flush=True)
path = snapshot_download(
    repo_id='Edge0/Audio8-ASR-Infinite',
    revision=revision,
    local_dir=destination,
)
print('model downloaded to:', path, flush=True)
''', MODEL_REVISION, MODEL_FOLDER)


In [ ]:
import gc
import json
import os
import struct

# release a whisper model if one was loaded in this notebook runtime.
if 'model' in globals():
    del model
gc.collect()
if 'torch' in sys.modules and sys.modules['torch'].cuda.is_available():
    sys.modules['torch'].cuda.empty_cache()

if not PYTHON.exists() or not MODEL_FOLDER.exists():
    raise RuntimeError('complete the environment and model download cells first')

actual_revision = run_command(['git', '-C', REPOSITORY, 'rev-parse', 'HEAD'])
if actual_revision != CODE_REVISION:
    raise RuntimeError('installed source revision differs; run cell 2 before loading')

parts = [
    ('model.safetensors', 'model-00001-of-00002.safetensors'),
    ('semantic_vad_heads.safetensors', 'model-00002-of-00002.safetensors'),
]
weight_map = {}
total_size = 0

for original_name, shard_name in parts:
    original = MODEL_FOLDER / original_name
    with original.open('rb') as file:
        header_size = struct.unpack('<Q', file.read(8))[0]
        if header_size > 100_000_000:
            raise RuntimeError('unexpected weight header size; check the downloaded file')
        header = json.loads(file.read(header_size))
    for name, info in header.items():
        if name == '__metadata__':
            continue
        if name in weight_map:
            raise RuntimeError(f'duplicate weight: {name}')
        weight_map[name] = shard_name
        start, end = info['data_offsets']
        total_size += end - start

expected_pause_keys = {
    f'semantic_vad_heads.{number}.{kind}'
    for number in range(4) for kind in ('weight', 'bias')
}
if not expected_pause_keys.issubset(weight_map):
    raise RuntimeError('the separate trained speech-pause weights are incomplete')

INDEXED_FOLDER.mkdir(exist_ok=True)
if (INDEXED_FOLDER / 'model.safetensors').exists():
    raise RuntimeError('prepared folder contains a main weight file that could bypass the index')

def link_existing(source, destination):
    if destination.exists():
        if not os.path.samefile(source, destination):
            raise RuntimeError(f'prepared file differs; inspect before replacing: {destination.name}')
    else:
        os.link(source, destination)

for original_name, shard_name in parts:
    link_existing(MODEL_FOLDER / original_name, INDEXED_FOLDER / shard_name)

for path in MODEL_FOLDER.iterdir():
    if (
        path.is_file()
        and not path.name.startswith('.')
        and not path.name.endswith('.safetensors')
        and not path.name.endswith('.safetensors.index.json')
    ):
        link_existing(path, INDEXED_FOLDER / path.name)

index = {'metadata': {'total_size': total_size}, 'weight_map': weight_map}
index_path = INDEXED_FOLDER / 'model.safetensors.index.json'
if index_path.exists():
    if json.loads(index_path.read_text()) != index:
        raise RuntimeError('prepared index differs; inspect it before replacing')
else:
    index_path.write_text(json.dumps(index, indent=2), encoding='utf-8')

print('prepared model folder:', INDEXED_FOLDER)
print('indexed weights:', len(weight_map))
print('indexed speech-pause weights:', len(expected_pause_keys))
print('testing loading; wait for the process to finish...', flush=True)

run_audio8('''
import sys
import torch
from audio8_asr_infinite.modeling.modeling_audio8_asr_infinite import (
    Audio8ASRInfiniteForConditionalGeneration,
)

if not torch.cuda.is_available():
    raise RuntimeError('no cuda device available')
x = torch.randn(128, 128, device='cuda', dtype=torch.float16)
y = x @ x
torch.cuda.synchronize()
del x, y

model = Audio8ASRInfiniteForConditionalGeneration.from_pretrained(
    sys.argv[1], torch_dtype=torch.float16,
).to('cuda').eval()
torch.cuda.synchronize()

print('model loaded successfully')
print('device:', next(model.parameters()).device)
print('dtype:', next(model.parameters()).dtype)
print('gpu memory allocated:', round(torch.cuda.memory_allocated() / 1024**3, 2), 'GiB')
''', INDEXED_FOLDER)
